# 14. 판정요지 문장 EDA 및 불용어 후보 정리

13번 실험에서 생성한 문장별 특징 데이터를 바탕으로 판정요지 문장의 길이·중복·어휘 분포를 탐색하고, HS 판정 맥락에 맞춘 불용어 후보를 정리합니다.

**중요:** 원문은 보존하고 불용어는 형태소 분석 후 모델 입력에만 적용합니다. '제외', '포함', '아닌', '혼합', '주성분', '용도'처럼 판단을 구체화하는 표현은 보호합니다.


In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('kiwipiepy') is None: subprocess.check_call([sys.executable,'-m','pip','-q','install','kiwipiepy'])
import re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from kiwipiepy import Kiwi
from urllib.parse import quote


## 1. 데이터 불러오기 및 기본 점검


In [ ]:
BASE = 'https://raw.githubusercontent.com/lin1118-code/HS-complete-product/main/'
cases_file = quote('완제품 모델 데이터_사례_전처리 후 - 사례_전처리 완료.csv',safe='')
cases = pd.read_csv(BASE+cases_file).fillna('')
from google.colab import files
import os
if not os.path.exists('13_sentence_features.csv'):
    print('13번 실험에서 생성한 13_sentence_features.csv 파일을 업로드해 주세요.')
    uploaded = files.upload()
    if '13_sentence_features.csv' not in uploaded: raise FileNotFoundError('13_sentence_features.csv 파일이 필요합니다.')
sentences = pd.read_csv('13_sentence_features.csv').fillna('')
print('문장 수:',len(sentences),'| 연결 가능한 사례 수:',len(cases))
display(sentences.head())


## 2. 문장 길이·중복·유사도 EDA


In [ ]:
sentences['char_len'] = sentences['sentence'].astype(str).str.len()
sentences['word_space_n'] = sentences['sentence'].astype(str).str.split().str.len()
sentences['is_duplicate'] = sentences['sentence'].astype(str).duplicated(keep=False)
print(sentences[['char_len','word_space_n','kb_similarity','attribute_similarity','specificity_score']].describe().round(3))
display(sentences.sort_values('char_len').head(15)[['row_index','sentence_id','char_len','sentence']])
display(sentences[sentences['is_duplicate']].sort_values('sentence')[['sentence','row_index']].head(30))
fig,axes = plt.subplots(1,3,figsize=(15,4))
sentences['char_len'].hist(ax=axes[0],bins=20); axes[0].set_title('문장 길이(문자 수)')
sentences['kb_similarity'].hist(ax=axes[1],bins=20); axes[1].set_title('KB 유사도')
sentences['attribute_similarity'].hist(ax=axes[2],bins=20); axes[2].set_title('제품 속성 유사도')
plt.tight_layout(); plt.show()


## 3. Kiwi 형태소 분석 및 품사별 빈도

명사·동사·형용사 중심으로 어휘 빈도를 확인합니다. 조사·어미 등 기능어는 별도로 살펴보고, 분류 판단에 중요한 부정·예외 표현은 제거하지 않습니다.


In [ ]:
kiwi = Kiwi()
def tokenize(text):
    return [(t.form,t.tag) for t in kiwi.tokenize(str(text))]
tokenized = [tokenize(x) for x in sentences['sentence']]
sentences['tokens'] = tokenized
content_tags = {'NNG','NNP','NNB','VV','VA','XR','SL','SN'}
content_terms = [form for row in tokenized for form,tag in row if tag in content_tags]
term_freq = pd.DataFrame(Counter(content_terms).most_common(),columns=['term','frequency'])
display(term_freq.head(100))
pos_freq = pd.DataFrame(Counter(tag for row in tokenized for _,tag in row).most_common(),columns=['pos','frequency'])
display(pos_freq)


## 4. HS 판정 맥락에 따른 불용어 후보

반복적으로 등장하는 일반 표현과 문서 형식 표현을 불용어 후보로 지정합니다. 아래 목록은 HS 판정요지 분석을 위한 프로젝트 기준이며, 최종 적용 전 모델 성능 비교를 거칩니다.


In [ ]:
boilerplate_candidates = {'것','경우','관세율표','규정','기재','등','따라서','물품','및','본건','본품','분류','설명','소호','수','이상','제품','함','해당','호','표','관세'}
protected_terms = {'아닌','아니','않','제외','포함','혼합','비혼합','주성분','성분','함량','용도','사용','적합','본질','특성','기능','형태','포장','소매','의약품','선스크린','선탠','입술','눈','피부'}
candidate_review = term_freq[term_freq['term'].isin(boilerplate_candidates)].copy()
candidate_review['protected'] = candidate_review['term'].isin(protected_terms)
display(candidate_review.sort_values('frequency',ascending=False))
print('불용어 후보:',sorted(boilerplate_candidates-protected_terms))
print('보호어:',sorted(protected_terms))


## 5. 문장 유형별 핵심 표현 탐색

부정·제외·포함·판정·성분·용도와 관련된 표현이 실제로 어떤 문맥에서 쓰이는지 원문 문장을 함께 확인합니다. 이 표현들은 모델이 학습해야 할 중요한 단서일 수 있으므로 불용어에서 보호합니다.


In [ ]:
pattern = r'제외|포함|아닌|아니|않|해당하|분류하|판단|주성분|함량|용도|사용하|혼합|본질|특성|적합'
cue_rows = sentences[sentences['sentence'].astype(str).str.contains(pattern,regex=True,na=False)][['row_index','sentence_id','sentence','kb_similarity','attribute_similarity']]
display(cue_rows.head(100))


## 6. 확정 후보 적용 전후 토큰 비교

사용자가 정리한 HS 판정요지 불용어 후보 22개를 적용합니다. 원문과 원래 토큰은 유지하고, 불용어 제거 토큰을 별도로 생성해 빈도 변화를 비교합니다. 형태소 단위의 완전 일치로 제거하므로 단어 일부를 문자열 치환하지 않습니다.


In [ ]:
stopwords_draft = boilerplate_candidates-protected_terms
def filter_tokens(items,stopwords):
    return [form for form,tag in items if tag in content_tags and form not in stopwords]
sentences['tokens_raw'] = [[form for form,tag in row if tag in content_tags] for row in tokenized]
sentences['tokens_filtered'] = [filter_tokens(row,stopwords_draft) for row in tokenized]
raw_freq = Counter(x for row in sentences['tokens_raw'] for x in row)
filtered_freq = Counter(x for row in sentences['tokens_filtered'] for x in row)
freq_compare = pd.DataFrame({'term':list(set(raw_freq)|set(filtered_freq))})
freq_compare['raw_frequency'] = freq_compare['term'].map(raw_freq).fillna(0).astype(int)
freq_compare['filtered_frequency'] = freq_compare['term'].map(filtered_freq).fillna(0).astype(int)
display(freq_compare.sort_values('raw_frequency',ascending=False).head(100))


## 7. 결과 저장

저장 파일은 EDA 결과와 사람이 검토할 불용어 후보·보호어 목록입니다. 검토 전에는 이 목록을 최종 학습 전처리에 적용하지 않습니다.


In [ ]:
sentences.drop(columns=['tokens'],errors='ignore').to_csv('14_sentence_eda.csv',index=False,encoding='utf-8-sig')
term_freq.to_csv('14_term_frequency.csv',index=False,encoding='utf-8-sig')
candidate_review.to_csv('14_stopword_candidates.csv',index=False,encoding='utf-8-sig')
freq_compare.to_csv('14_stopword_before_after.csv',index=False,encoding='utf-8-sig')
with open('14_stopword_draft.json','w',encoding='utf-8') as f: json.dump({'stopwords_draft':sorted(stopwords_draft),'protected_terms':sorted(protected_terms)},f,ensure_ascii=False,indent=2)
print('14번 EDA 결과 저장 완료')


## 8. 해석 및 다음 단계

1. HS 판정요지 분석을 위한 불용어 후보 22개를 적용해 토큰 빈도 변화를 확인합니다.
2. '제외·포함·아닌·않다' 등 부정·예외 표현과 성분·용도 관련 표현은 보호합니다.
3. 불용어 적용은 형태소 단위 완전 일치로 수행하며 원문은 보존합니다.
4. 다음 단계에서 불용어 적용 전후의 TF-IDF+SVM 분류 성능을 동일한 검증 분할로 비교한 뒤 최종 목록을 확정합니다.
